# Notebook 08 — Discounted Cash Price (general-customer rates) · code 73721

**Why this notebook exists.** The negotiated-rate ladder (notebook 06) is *payer-keyed* — every row hangs off a payer/LOB. The **discounted cash price** has no payer, so `query_procedure_rates_agg` groups it away and it never reaches `all_v4`. To see what a general / self-pay customer actually pays, we have to read it **raw** from the DuckDBs.

**Goal.** Per-hospital discounted cash price for the knee MRI (73721, outpatient), set against the **gross / chargemaster list price** that rides on the same rows.

**The two consumer numbers**
- *Discounted cash price* — what an uninsured / self-pay patient actually pays. The real general-customer rate.
- *Gross charge* — the list/sticker price almost nobody pays; here it's the reference for the markup story.

**Discipline.** Schema is unknown, so Step 1 reads it off the DuckDB and Step 2 sets the real column names. Nothing downstream assumes a column name that hasn't been confirmed.

## Step 0 — config & locate the DuckDBs

Edit `DB_GLOB` to point at your five `.duckdb` files. If `DBs found:` comes back empty, the pattern is wrong.

In [2]:
import duckdb, glob, os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DB_GLOB = r'C:\Users\kedha\Documents\dfw-hospital-pricing\data\raw\*.duckdb'
CODE    = '73721'
SETTING = 'outpatient'

db_paths = sorted(glob.glob(DB_GLOB))
print('DBs found:', db_paths)
assert db_paths, f'No DuckDBs matched {DB_GLOB!r} -- fix DB_GLOB above'

DBs found: ['C:\\Users\\kedha\\Documents\\dfw-hospital-pricing\\data\\raw\\baylor_university_medical_center-69947_parsed.duckdb', 'C:\\Users\\kedha\\Documents\\dfw-hospital-pricing\\data\\raw\\medical_city_alliance_hospital-77912_parsed.duckdb', 'C:\\Users\\kedha\\Documents\\dfw-hospital-pricing\\data\\raw\\methodist_dallas_medical_center-6000b_parsed.duckdb', 'C:\\Users\\kedha\\Documents\\dfw-hospital-pricing\\data\\raw\\parkland_health-6e88d_parsed.duckdb', 'C:\\Users\\kedha\\Documents\\dfw-hospital-pricing\\data\\raw\\texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb']


## Step 1 — read the schema

Pull the table list and column names straight off the first DuckDB. Scan the output for the **discounted-cash** and **gross/list** columns, plus the **code** and **setting** columns. You'll feed the real names into Step 2.

In [ ]:
con = duckdb.connect(db_paths[0], read_only=True)

print('Tables:', [t[0] for t in con.execute('SHOW TABLES').fetchall()], '\n')
for (tbl,) in con.execute('SHOW TABLES').fetchall():
    print(f'=== {tbl} ===')
    print(con.execute(f'DESCRIBE {tbl}').fetchdf()[['column_name', 'column_type']].to_string())
    print()
con.close()

## Step 2 — set the real names

Fill these in from the `DESCRIBE` output above. The defaults are common CMS-MRF names — **verify each against the actual columns; don't trust them.** Set `SETTING_COL = None` if there's no inpatient/outpatient column.

In [ ]:
# >>> set from the DESCRIBE output above <<<
TABLE       = 'rates'             # raw table holding the procedure rows
CODE_COL    = 'code'              # procedure / billing code
SETTING_COL = 'setting'           # inpatient/outpatient column; None if absent
CASH_COL    = 'discounted_cash'   # discounted cash / self-pay price
GROSS_COL   = 'gross_charge'      # gross / chargemaster list price

## Step 3 — pull cash + gross, one row per hospital

No payer grouping. The column check turns a wrong name into a clear message (listing what's available) instead of a cryptic error. Hospital name is taken from the DuckDB filename — adjust the mapping if your files aren't named by hospital.

In [ ]:
def pull_cash_gross(path):
    hosp = os.path.splitext(os.path.basename(path))[0]
    c = duckdb.connect(path, read_only=True)

    cols = set(c.execute(f'DESCRIBE {TABLE}').fetchdf()['column_name'])
    needed = [CODE_COL, CASH_COL, GROSS_COL] + ([SETTING_COL] if SETTING_COL else [])
    missing = [x for x in needed if x not in cols]
    if missing:
        raise KeyError(f'{hosp}: missing {missing}. Available columns: {sorted(cols)}')

    where = f"CAST({CODE_COL} AS VARCHAR) = '{CODE}'"
    if SETTING_COL:
        where += f" AND lower(CAST({SETTING_COL} AS VARCHAR)) LIKE '%{SETTING.lower()}%'"

    q = f'''
        SELECT DISTINCT
            {CASH_COL}  AS discounted_cash,
            {GROSS_COL} AS gross_charge
        FROM {TABLE}
        WHERE {where}
    '''
    df = c.execute(q).fetchdf()
    c.close()
    df.insert(0, 'hospital', hosp)
    return df

cash = pd.concat([pull_cash_gross(p) for p in db_paths], ignore_index=True)
print(cash.to_string(index=False))

## Step 4 — the general-customer view

Median cash and gross per hospital, with cash as a share of gross and the gross-to-cash markup. A hospital showing `NaN` cash hasn't published a self-pay price — that's a transparency finding, not a bug. (Recall MCA's gross came through `NaN` in the negotiated agg, and Parkland is effectively Medicaid-only.)

In [ ]:
view = (cash
        .groupby('hospital', as_index=False)
        .agg(discounted_cash=('discounted_cash', 'median'),
             gross_charge=('gross_charge', 'median')))

view['cash_pct_of_gross']      = (view['discounted_cash'] / view['gross_charge'] * 100).round(1)
view['gross_to_cash_multiple'] = (view['gross_charge'] / view['discounted_cash']).round(1)
view = view.sort_values('discounted_cash').reset_index(drop=True)

print(view.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plot = view.dropna(subset=['discounted_cash'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(plot['hospital'], plot['discounted_cash'])
ax.set_ylabel('Discounted cash price ($)')
ax.set_title(f'Discounted cash price for {CODE} ({SETTING}) by hospital')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(plot['discounted_cash']):
    ax.text(i, v, f'${v:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Notes & next steps

- This view deliberately **bypasses `query_procedure_rates_agg`** — that function is payer-grouped and drops the cash price. Everything here reads raw.
- **Watch for missing cash prices.** A `NaN` is itself a result about how transparent (or not) a hospital is being.
- **Overlay against the LOB ladder.** The interesting question: where does the cash price sit relative to commercial / Medicare / Medicaid negotiated rates? (Often cash lands *between* Medicare and commercial — sometimes below commercial, which is the headline.)
- **Markup story.** `gross_to_cash_multiple` is the consumer-facing version of the ~21× gross-vs-negotiated gap we saw on the COVID row.
- **Basket extension.** Everything is parameterized by `CODE` — swapping in another shoppable code is a one-line change.